### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know. Parrots are birds known for mimicking human speech. I think it's part of their social behavior. Maybe they learned it from their parents or other parrots in the wild. But why do they do it? Is it just imitation, or is there more to it?\n\nFirst, I should consider their anatomy. I remember that birds have a syrinx, which is their vocal organ, located at the base of the trachea. The syrinx allows them to produce a wide range of sounds. But how does that enable them to mimic human speech? Maybe the structure of the syrinx in parrots is more complex, giving them finer control over sounds.\n\nThen there's the brain. I think parrots have a well-developed part of the brain related to vocal learning, similar to humans. The area might be called the song system, which includes regions like the HVC (used as a proper name) and the robust nucleus of the arcopallium. These areas are

In [7]:
from langchain.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location"""
    # In a real implementation, you would call a weather API here
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_current_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EDF7973620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EDF8AE4440>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_current_weather', 'description': 'Get the current weather in 

In [9]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_current_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I'll call the function with Boston as the argument. Make sure the JSON is correctly formatted with the name and arguments. No other functions are available, so this should be straightforward.\n", 'tool_calls': [{'id': 'n1smq9306', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 157, 'total_tokens': 259, 'completion_time': 0.175467653, 'completion_tokens_details': {'reasoning_tokens': 77}, 'prompt_time': 0.006318528, 'prompt_tokens_details': None, 'queue_time': 0.051009221, 'total_time': 0.181786181}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reaso

### tool execution looop

In [17]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_current_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is sunny."

The current weather in Boston is sunny with a temperature of 25°C.


In [16]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Boston. I need to use the get_current_weather function. Let me check the function parameters. It requires a location, which in this case is Boston. I should structure the tool call with the name of the function and the arguments as a JSON object. Make sure the location is a string. Alright, that's straightforward. Let me put that into the correct JSON format inside the tool_call tags.\n", 'tool_calls': [{'id': 'k8ftrz7ex', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_current_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 156, 'total_tokens': 271, 'completion_time': 0.198540213, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.006218959, 'prompt_tokens_details': None, 'queue_time': 0.04979097, 'total_time': 0.2047